# RL-token-minimize — full pipeline on Kaggle

Runs the project end to end on a Kaggle GPU: data prep → baseline eval → LoRA SFT → post-SFT eval → smoke RL.

**Setup**: Settings → Accelerator → GPU (T4 or P100). Internet must be enabled (Settings → Internet → On) to clone the repo and download the model/dataset. On T4 ×2 only the first GPU is used — HF Trainer's DataParallel wrapping breaks with the PEFT setup, so the setup cell pins `CUDA_VISIBLE_DEVICES=0`.

Artifacts land in `/kaggle/working/RL-token-minimize/checkpoints/` — download them from the Output tab.

In [1]:
%env CUDA_VISIBLE_DEVICES=0
%env PYTORCH_ALLOC_CONF=expandable_segments:True
%cd /kaggle/working
!rm -rf RL-token-minimize
!git clone --depth 1 https://github.com/augustoFranke/RL-token-minimize.git
%cd RL-token-minimize
!mkdir -p checkpoints
!pip uninstall -q -y torchao
!pip install -q -U "transformers>=5.13" "trl>=1.7.1" "peft>=0.19.1" "datasets>=5.0.0" "accelerate>=1.14.0" pytest
!pip install -q -e . --no-deps
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

env: CUDA_VISIBLE_DEVICES=0
env: PYTORCH_ALLOC_CONF=expandable_segments:True
/kaggle/working
Cloning into 'RL-token-minimize'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 34 (delta 0), reused 23 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 132.81 KiB | 16.60 MiB/s, done.
/kaggle/working/RL-token-minimize
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installi

In [2]:
!python -m pytest -q

.................................................................        [100%]
65 passed in 3.56s


In [3]:
!python scripts/prepare_data.py
!wc -l data/*.jsonl

README.md: 12.5kB [00:00, 25.0MB/s]
python/test-00000-of-00001.parquet: 100%|█████| 191k/191k [00:00<00:00, 431kB/s]
Generating test split: 100%|█████████| 164/164 [00:00<00:00, 2665.33 examples/s]
kept 161 tasks (121 train / 40 eval)
skipped: {'buggy_passes': 0, 'fixed_fails': 3, 'no_clean_trace': 0}
   121 data/sft_train.jsonl
    40 data/tasks_eval.jsonl
   121 data/tasks_train.jsonl
   282 total


In [4]:
!python scripts/evaluate.py --limit 5 --out checkpoints/eval_base5.jsonl

config.json: 100%|█████████████████████████████| 726/726 [00:00<00:00, 3.23MB/s]
tokenizer_config.json: 9.73kB [00:00, 16.3MB/s]
vocab.json: 2.78MB [00:00, 102MB/s]
merges.txt: 1.67MB [00:00, 116MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 26.8MB/s]
model.safetensors: 100%|████████████████████| 1.50G/1.50G [00:05<00:00, 268MB/s]
generation_config.json: 100%|██████████████████| 239/239 [00:00<00:00, 1.17MB/s]
[1/5] Python_99: fail tokens=40
[2/5] Python_12: fail tokens=40
[3/5] Python_143: fail tokens=66
[4/5] Python_29: fail tokens=92
[5/5] Python_19: fail tokens=79
{
  "n": 5,
  "adapter": null,
  "thinking": false,
  "pass_rate": 0.0,
  "mean_reward": -0.9,
  "mean_tokens": 63.4,
  "mean_thinking_tokens": 0.0,
  "mean_tool_calls": 1.0,
  "protocol_error_rate": 0.8
}


In [5]:
!python scripts/train_sft.py

Tokenizing train dataset: 100%|███████| 121/121 [00:00<00:00, 373.32 examples/s]
Building labels for train dataset: 100%|█| 121/121 [00:00<00:00, 1009.35 example
Truncating train dataset: 100%|███████| 121/121 [00:00<00:00, 857.41 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
{'loss': '1.56', 'grad_norm': '1.22', 'learning_rate': '9.167e-05', 'entropy': '0.9192', 'num_tokens': '3.939e+04', 'mean_token_accuracy': '0.7139', 'epoch': '0.3306'}
{'loss': '1.006', 'grad_norm': '0.8559', 'learning_rate': '8.125e-05', 'entropy': '1.005', 'num_tokens': '7.935e+04', 'mean_token_accuracy': '0.7815', 'epoch': '0.6612'}
{'loss': '0.7096', 'grad_norm': '1.065', 'learning_rate': '7.083e-05', 'entropy': '0.8213', 'num_tokens': '1.203e+05', 'mean_token_acc

In [6]:
!python scripts/evaluate.py --adapter checkpoints/sft --limit 5 --out checkpoints/eval_sft5.jsonl

Loading weights: 100%|███████████████████████| 311/311 [00:00<00:00, 396.39it/s]
[1/5] Python_99: fail tokens=147
[2/5] Python_12: fail tokens=74
[3/5] Python_143: fail tokens=98
[4/5] Python_29: PASS tokens=92
[5/5] Python_19: fail tokens=100
{
  "n": 5,
  "adapter": "checkpoints/sft",
  "thinking": false,
  "pass_rate": 0.2,
  "mean_reward": 0.0355078125,
  "mean_tokens": 102.2,
  "mean_thinking_tokens": 0.0,
  "mean_tool_calls": 2.0,
  "protocol_error_rate": 0.0
}


In [7]:
!python scripts/train_rl.py --no-thinking --steps 10 --output checkpoints/rl_smoke
!tail -3 checkpoints/rl_smoke/log.jsonl

Loading weights: 100%|███████████████████████| 311/311 [00:00<00:00, 398.93it/s]
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 0, 'mean_reward': -0.09999999999999999, 'pass_rate': 0.0, 'mean_tokens': 233.5, 'mean_thinking_tokens': 0.0}
{'step': 1, 'mean_reward': 0.2, 'pass_rate': 0.0, 'mean_tokens': 115.5, 'mean_thinking_tokens': 0.0}
{'step': 2, 'mean_reward': 0.816417236328125, 'pass_rate': 0.5, 'mean_tokens': 114.0, 'mean_thinking_tokens': 0.0}
{'step': 3, 'mean_reward': 0.182708740234375, 'pass_rate': 0.125, 'mean_tokens': 164.125, 'mean_thinking_tokens': 0.0}
{'step': 4, 'mean_reward': 0.09688720703125, 'pass_rate': 0.125, 'mean_tokens': 183.75, 'mean_thinking_tokens': 0.0}
{'step': 5, 'mean_reward': 0.09761962890625, 'pass_rate': 0.125, 'mean_tokens': 93.25, 'mean_thinking_tokens': 0.0}
{'step': 6, 'mean_reward': -0.06249999999999999, 'pass_rate': 0.0, 'mean_tokens': 231.875, 'mean_thinking_tokens': 0.0}
{'step': 

## Full runs (uncomment once the smoke run looks healthy)

```
!python scripts/evaluate.py --adapter checkpoints/sft --out checkpoints/eval_sft_full.jsonl
!python scripts/train_rl.py --no-thinking --output checkpoints/rl_run1
!python scripts/train_rl.py --thinking --token-cost all --output checkpoints/rl_run2
!python scripts/train_rl.py --thinking --token-cost final --output checkpoints/rl_run3
!python scripts/evaluate.py --adapter checkpoints/rl_run1/final --out checkpoints/eval_rl1.jsonl
```

In [8]:
!python scripts/evaluate.py --out checkpoints/eval_base_full.jsonl

Loading weights: 100%|███████████████████████| 311/311 [00:00<00:00, 436.75it/s]
[1/40] Python_99: fail tokens=40
[2/40] Python_12: fail tokens=40
[3/40] Python_143: fail tokens=66
[4/40] Python_29: fail tokens=92
[5/40] Python_19: fail tokens=79
[6/40] Python_134: fail tokens=131
[7/40] Python_112: fail tokens=77
[8/40] Python_14: fail tokens=97
[9/40] Python_41: fail tokens=93
[10/40] Python_53: fail tokens=95
[11/40] Python_61: fail tokens=86
[12/40] Python_104: fail tokens=219
[13/40] Python_16: fail tokens=97
[14/40] Python_33: fail tokens=40
[15/40] Python_163: fail tokens=94
[16/40] Python_70: fail tokens=43
[17/40] Python_86: fail tokens=80
[18/40] Python_159: fail tokens=140
[19/40] Python_57: fail tokens=219
[20/40] Python_97: fail tokens=67
[21/40] Python_148: fail tokens=82
[22/40] Python_113: fail tokens=146
[23/40] Python_92: fail tokens=76
[24/40] Python_136: fail tokens=60
[25/40] Python_3: fail tokens=107
[26/40] Python_145: fail tokens=82
[27/40] Python_71: fail token

In [9]:
!python scripts/evaluate.py --adapter checkpoints/sft --out checkpoints/eval_sft_full.jsonl

Loading weights: 100%|███████████████████████| 311/311 [00:00<00:00, 372.11it/s]
[1/40] Python_99: fail tokens=147
[2/40] Python_12: fail tokens=74
[3/40] Python_143: fail tokens=98
[4/40] Python_29: PASS tokens=92
[5/40] Python_19: fail tokens=100
[6/40] Python_134: fail tokens=116
[7/40] Python_112: fail tokens=88
[8/40] Python_14: PASS tokens=87
[9/40] Python_41: fail tokens=81
[10/40] Python_53: fail tokens=86
[11/40] Python_61: fail tokens=80
[12/40] Python_104: fail tokens=82
[13/40] Python_16: PASS tokens=83
[14/40] Python_33: fail tokens=74
[15/40] Python_163: fail tokens=110
[16/40] Python_70: fail tokens=80
[17/40] Python_86: fail tokens=104
[18/40] Python_159: fail tokens=94
[19/40] Python_57: fail tokens=74
[20/40] Python_97: fail tokens=110
[21/40] Python_148: fail tokens=1044
[22/40] Python_113: fail tokens=93
[23/40] Python_92: fail tokens=112
[24/40] Python_136: fail tokens=96
[25/40] Python_3: fail tokens=430
[26/40] Python_145: fail tokens=96
[27/40] Python_71: fail t

In [10]:
!python scripts/train_rl.py --no-thinking --output checkpoints/rl_run1
!tail -5 checkpoints/rl_run1/log.jsonl

Loading weights: 100%|███████████████████████| 311/311 [00:00<00:00, 354.72it/s]
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
{'step': 0, 'mean_reward': 0.097283935546875, 'pass_rate': 0.125, 'mean_tokens': 192.625, 'mean_thinking_tokens': 0.0}
{'step': 1, 'mean_reward': 0.11250000000000002, 'pass_rate': 0.0, 'mean_tokens': 134.5, 'mean_thinking_tokens': 0.0}
{'step': 2, 'mean_reward': 0.53870849609375, 'pass_rate': 0.5, 'mean_tokens': 344.0, 'mean_thinking_tokens': 0.0}
{'step': 3, 'mean_reward': -0.09999999999999999, 'pass_rate': 0.0, 'mean_tokens': 124.125, 'mean_thinking_tokens': 0.0}
{'step': 4, 'mean_reward': 0.192767333984375, 'pass_rate': 0.25, 'mean_tokens': 144.375, 'mean_thinking_tokens': 0.0}
{'step': 5, 'mean_reward': 0.2, 'pass_rate': 0.0, 'mean_tokens': 102.875, 'mean_thinking_tokens': 0.0}
{'step': 6, 'mean_reward': 0.05000000000000001, 'pass_rate': 0.0, 'mean_tokens': 133.0, 'mean_thinking_tokens': 0.0}
{'step'